# Proyecto Integrador 2 · Parte II — Clustering y detección de anomalías
## Tareas 8 a 11, 13 y 14: tipos de proveedores y de entidades

**Pregunta de interés P1:** *¿Qué tipos de proveedores existen según cómo contratan con el Estado, y cuáles se salen de su tipo?* El mismo método se aplica a las **entidades compradoras**.

**Qué hace este notebook**
1. Estandariza las características (logaritmo en lo que tiene colas largas) y las reduce con **PCA**.
2. **KMeans++** con varios *k*. El *k* se elige con el **codo** de la inercia (como en clase) y se revisa con la **silueta** y con la **estabilidad** de los grupos.
3. **DBSCAN y HDBSCAN** (por densidad) sobre una muestra: dónde hay grupos y qué queda como ruido.
4. **Jerárquico** (Ward) sobre una muestra, para ver un dendrograma.
5. Interpreta los grupos y los **casos extremos** (calculados con todos los puntos, no solo con la muestra).

**Por qué sklearn y no Spark:** los vectores del notebook 1 ya son pequeños (unas decenas de miles de filas × 9 columnas) y caben en memoria. Spark añadiría tiempo de arranque y de coordinación sin ganar velocidad. DBSCAN y HDBSCAN van en local sobre una muestra, como permite el enunciado.

**Ética:** un caso extremo es una **señal descriptiva para revisar**, no una acusación. Un proveedor muy grande o muy concentrado puede ser perfectamente legítimo.

## 0. Configuración

In [1]:
import os, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, HDBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram

RUTA_P2 = os.environ.get("P2_DATOS", "/content/drive/MyDrive/proyecto_integrador_seace/p2")
WEB     = os.environ.get("P2_WEB",   "/content/drive/MyDrive/proyecto_integrador_seace/web_artefactos/p2")
os.makedirs(WEB, exist_ok=True)

SEED = 42
KS = list(range(2, 11))      # valores de k que se prueban
K_MIN_ELEGIBLE = 3           # k = 2 casi siempre solo separa "grandes" de "pequeños"; para comparar silueta se mira desde 3
K_ELEGIDO = {"proveedor": 5, "entidad": 6}   # k elegido por el codo, revisado con silueta y estabilidad (secciones 2b y 2c)
REPETICIONES = 20            # pares de submuestras para medir la estabilidad
N_MUESTRA = 5000             # puntos para los métodos de densidad y el gráfico
N_JERARQUICO = 1500          # puntos para el dendrograma
VARIANZA_PCA = 0.90          # PCA se queda con las componentes que expliquen al menos 90%

# Qué columnas usa cada análisis. "log" = columnas con colas largas, a las que se les aplica log(1+x).
SUJETOS = {
    "proveedor": dict(ruta="features_proveedor", id="ruc", filtro="n_participaciones >= 5",
        num=["n_participaciones", "tasa_exito", "n_adjudicaciones", "monto_total", "monto_mediano",
             "n_entidades", "n_departamentos", "concentracion", "meses_activo"],
        log=["n_participaciones", "n_adjudicaciones", "monto_total", "monto_mediano", "n_entidades", "n_departamentos", "meses_activo"],
        desc=["categoria_principal"]),
    "entidad": dict(ruta="features_entidad", id="entidad_id", filtro="n_procesos >= 5",
        num=["n_procesos", "n_proveedores", "monto_total", "monto_mediano", "pct_bienes", "pct_servicios", "pct_obras",
             "postores_promedio", "concentracion"],
        log=["n_procesos", "n_proveedores", "monto_total", "monto_mediano", "postores_promedio"],
        desc=["entidad_tipo", "departamento"]),
}

def proteger(df):
    # datos personales (como en el Proyecto 1): RUC que no empieza con 20 va enmascarado; persona natural sin nombre
    if not {"sujeto", "id", "nombre"} <= set(df.columns): return df
    df = df.copy(); ids = df.id.astype(str); es_prov = df.sujeto == "proveedor"
    df["nombre"] = df["nombre"].astype(str).str.replace(chr(92) + '"', '"', regex=False)   # quita barras invertidas heredadas de la fuente
    mask = es_prov & ~ids.str.startswith("20"); natural = es_prov & ids.str[:2].isin(["10", "15", "16", "17"])
    enm = ids.str[:5] + "****" + ids.str[-2:]
    df.loc[natural, "nombre"] = "Persona natural (RUC " + enm[natural] + ")"
    df.loc[mask, "id"] = enm[mask]
    return df

def guardar_csv(nombre, df):
    df = proteger(df)
    df.to_csv(f"{WEB}/{nombre}", index=False)
    print("guardado:", nombre, len(df), "filas")

## 1. Preparación: filtro, logaritmo, estandarizar y PCA
- **Filtro:** proveedores con al menos 5 participaciones y entidades con al menos 5 procesos. Con menos, una tasa o un promedio es ruido (ganar 1 de 1 procesos da "100% de éxito").
- **Logaritmo:** montos y conteos tienen colas muy largas (unos pocos gigantes). `log(1+x)` evita que ellos dominen la distancia.
- **Estandarizar:** cada columna queda con media 0 y desviación 1, para que ninguna pese más solo por su unidad (soles frente a proporción).
- **PCA:** resume las columnas en pocas **componentes** (combinaciones de ellas) que no se repiten entre sí. Nos quedamos con las que expliquen al menos 90% de la variación.

In [2]:
D = {}   # aquí se guarda todo, por sujeto
for nombre, cfg in SUJETOS.items():
    df = pd.read_parquet(f"{RUTA_P2}/{cfg['ruta']}").query(cfg["filtro"]).reset_index(drop=True)
    X = df[cfg["num"]].copy()
    X[cfg["log"]] = np.log1p(X[cfg["log"]])
    Z = StandardScaler().fit_transform(X)
    pca = PCA(n_components=VARIANZA_PCA, random_state=SEED).fit(Z)
    D[nombre] = dict(df=df, Z=Z, pca=pca, P=pca.transform(Z))
    print(f"{nombre}: {len(df):,} filas | {pca.n_components_} componentes explican {100*pca.explained_variance_ratio_.sum():.1f}%")

filas = []
for nombre, d in D.items():
    var = d["pca"].explained_variance_ratio_
    for i, v in enumerate(var):
        filas.append({"sujeto": nombre, "componente": i + 1, "var_explicada": round(v, 4), "var_acumulada": round(var[:i+1].sum(), 4)})
pca_varianza = pd.DataFrame(filas)
pca_varianza

proveedor: 43,115 filas | 4 componentes explican 95.1%
entidad: 2,679 filas | 5 componentes explican 95.6%


,sujeto,componente,var_explicada,var_acumulada
0,proveedor,1,0.6363,0.6363
1,proveedor,2,0.1575,0.7939
2,proveedor,3,0.0830,0.8769
3,proveedor,4,0.0736,0.9505
4,entidad,1,0.4194,0.4194
5,entidad,2,0.1819,0.6013
6,entidad,3,0.1486,0.7499
7,entidad,4,0.1240,0.8739
8,entidad,5,0.0824,0.9564


In [3]:
# ¿Qué columnas pesan más en las dos primeras componentes? (las que se dibujan en el gráfico)
for nombre, d in D.items():
    cargas = pd.DataFrame(d["pca"].components_[:2].T, index=SUJETOS[nombre]["num"], columns=["CP1", "CP2"]).round(2)
    print(f"--- {nombre} ---"); print(cargas.reindex(cargas.CP1.abs().sort_values(ascending=False).index).to_string())

--- proveedor ---
                    CP1   CP2
monto_total        0.41 -0.09
monto_mediano      0.40 -0.12
n_adjudicaciones   0.39  0.05
n_departamentos    0.39  0.06
n_entidades        0.38  0.13
concentracion      0.30 -0.31
tasa_exito         0.25 -0.45
meses_activo       0.19  0.43
n_participaciones  0.18  0.68
--- entidad ---
                    CP1   CP2
n_proveedores      0.49 -0.05
n_procesos         0.45 -0.01
monto_total        0.45  0.00
concentracion     -0.36  0.07
pct_obras         -0.31 -0.05
monto_mediano      0.21  0.03
postores_promedio  0.20  0.26
pct_bienes         0.16 -0.67
pct_servicios      0.13  0.69


## 2. KMeans++ y elección de k
**KMeans++** reparte los puntos en *k* grupos alrededor de centros; el "++" elige bien los centros iniciales. Para elegir *k* miramos dos cosas:
- **Inercia** (codo): suma de distancias al centro de cada grupo. Siempre baja al subir *k*; se busca el punto donde deja de bajar rápido.
- **Silueta:** qué tan bien separado está cada punto de los otros grupos (de −1 a 1; más alto es mejor). Se calcula con 5,000 puntos al azar para que no tarde.

**Regla:** el *k* se lee del **codo** (es lo que pide el profe). La **silueta** y la **estabilidad** (sección 2c) sirven para revisar esa elección. El codo no siempre es nítido, por eso el *k* **no es automático**: se fija en `K_ELEGIDO` y se justifica con los números de las secciones 2b y 2c.

In [4]:
filas, perfiles = [], []
for nombre, d in D.items():
    cfg, df, P = SUJETOS[nombre], d["df"], d["P"]
    d["km"] = {}
    for k in KS:
        km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=SEED).fit(P)
        sil = silhouette_score(P, km.labels_, sample_size=5000, random_state=SEED)
        d["km"][k] = km
        filas.append({"sujeto": nombre, "k": k, "inercia": round(km.inertia_, 1), "silueta": round(sil, 4)})
        med = df.groupby(km.labels_)[cfg["num"]].median()
        n = pd.Series(km.labels_).value_counts().sort_index()
        for c in med.index:
            for col in cfg["num"]:
                perfiles.append({"sujeto": nombre, "k": k, "cluster": int(c), "n": int(n[c]), "caracteristica": col, "mediana": round(float(med.loc[c, col]), 4)})

kmeans_k = pd.DataFrame(filas)
kmeans_k["elegido"] = False
for nombre, k in K_ELEGIDO.items():
    kmeans_k.loc[(kmeans_k.sujeto == nombre) & (kmeans_k.k == k), "elegido"] = True
    D[nombre]["k"] = k
kmeans_perfiles = pd.DataFrame(perfiles)
print(kmeans_k.pivot(index="k", columns="sujeto", values=["inercia", "silueta"]).round(3).to_string())
print("k elegido:", {n: d["k"] for n, d in D.items()})
for nombre, d in D.items():
    t = kmeans_k[(kmeans_k.sujeto == nombre) & (kmeans_k.k >= K_MIN_ELEGIBLE)]
    mejor, el = t.loc[t.silueta.idxmax()], t[t.k == d["k"]].iloc[0]
    print(f"{nombre}: silueta del k elegido ({d['k']}) = {el.silueta:.3f} | mejor silueta desde k = {K_MIN_ELEGIBLE}: k = {int(mejor.k)} ({mejor.silueta:.3f})")

        inercia           silueta          
sujeto  entidad proveedor entidad proveedor
k                                          
2       16641.6  141331.4   0.264     0.597
3       14623.7  104067.5   0.241     0.536
4       11275.9   82411.2   0.230     0.463
5        9283.5   62589.7   0.251     0.462
6        8208.0   52933.8   0.252     0.439
7        7399.7   47186.1   0.241     0.423
8        6885.2   41451.7   0.228     0.356
9        6461.0   37504.6   0.220     0.360
10       6094.7   34363.8   0.209     0.367
k elegido: {'proveedor': 5, 'entidad': 6}
proveedor: silueta del k elegido (5) = 0.462 | mejor silueta desde k = 3: k = 3 (0.536)
entidad: silueta del k elegido (6) = 0.252 | mejor silueta desde k = 3: k = 6 (0.252)


## 2b. El codo de la inercia, con números
La inercia siempre baja al subir *k*. El **codo** está donde la mejora se frena. Para no depender solo de la vista se calcula, para cada *k*, **cuánto baja la inercia** respecto al *k* anterior (en valor y en %) y el **ritmo**: la bajada de este *k* dividida entre la del anterior (menor que 1 = la mejora se frena).

In [5]:
for nombre in D:
    t = kmeans_k[kmeans_k.sujeto == nombre].sort_values("k").copy()
    t["baja_inercia"] = -t.inercia.diff()
    t["baja_%"] = (100 * t.baja_inercia / t.inercia.shift(1)).round(1)
    t["ritmo_vs_anterior"] = (t.baja_inercia / t.baja_inercia.shift(1)).round(2)   # <1: la mejora se frena
    kmeans_k.loc[t.index, "baja_inercia"] = t.baja_inercia.round(1)
    kmeans_k.loc[t.index, "baja_pct"] = t["baja_%"]
    print(f"--- {nombre} ---"); print(t[["k", "inercia", "baja_inercia", "baja_%", "ritmo_vs_anterior", "silueta"]].round(3).to_string(index=False)); print()

--- proveedor ---
 k  inercia  baja_inercia  baja_%  ritmo_vs_anterior  silueta
 2 141331.4           NaN     NaN                NaN    0.597
 3 104067.5       37263.9    26.4                NaN    0.536
 4  82411.2       21656.3    20.8               0.58    0.463
 5  62589.7       19821.5    24.1               0.92    0.462
 6  52933.8        9655.9    15.4               0.49    0.439
 7  47186.1        5747.7    10.9               0.60    0.423
 8  41451.7        5734.4    12.2               1.00    0.356
 9  37504.6        3947.1     9.5               0.69    0.360
10  34363.8        3140.8     8.4               0.80    0.367

--- entidad ---
 k  inercia  baja_inercia  baja_%  ritmo_vs_anterior  silueta
 2  16641.6           NaN     NaN                NaN    0.264
 3  14623.7        2017.9    12.1                NaN    0.241
 4  11275.9        3347.8    22.9               1.66    0.230
 5   9283.5        1992.4    17.7               0.60    0.251
 6   8208.0        1075.5    11.6  

### ¿Qué cambia si se usa otro *k*? (proveedores)
Como el codo no es nítido, se compara el perfil de los grupos con *k* = 3, 4 y 5. Si los tipos principales se mantienen, la conclusión no depende de esa elección.

In [6]:
cfg = SUJETOS["proveedor"]
for k in (3, 4, 5):
    p = kmeans_perfiles[(kmeans_perfiles.sujeto == "proveedor") & (kmeans_perfiles.k == k)]
    t = p.pivot(index="cluster", columns="caracteristica", values="mediana")[["n_participaciones", "tasa_exito", "n_adjudicaciones", "n_entidades", "concentracion", "meses_activo"]]
    t.insert(0, "n", p.groupby("cluster").n.first())
    print(f"=== proveedores, k = {k} ==="); print(t.round(2).to_string()); print()

=== proveedores, k = 3 ===
caracteristica      n  n_participaciones  tasa_exito  n_adjudicaciones  n_entidades  concentracion  meses_activo
cluster                                                                                                         
0               21846               11.0        0.00               0.0          0.0           0.00          16.1
1               14783               14.0        0.13               2.0          1.0           1.00          22.5
2                6486               90.0        0.08               7.0          4.0           0.33          28.5

=== proveedores, k = 4 ===
caracteristica      n  n_participaciones  tasa_exito  n_adjudicaciones  n_entidades  concentracion  meses_activo
cluster                                                                                                         
0                5670                7.0        0.00               0.0          0.0           0.00           4.0
1               14777               14.0 

## 2c. ¿Los grupos son estables?
Un grupo es creíble si **aparece igual** aunque cambien los datos. Prueba: se parten los puntos al azar en dos mitades, se ajusta KMeans++ en cada mitad por separado, y cada modelo etiqueta a **todos** los puntos. Se compara el parecido de las dos etiquetas con el **índice de Rand ajustado (ARI)**: 1 = los mismos grupos, 0 = tanto como el azar. Se repite `REPETICIONES` veces y se resume con el promedio y el peor caso. Es el "otro criterio" que permite el enunciado.

In [7]:
from sklearn.metrics import adjusted_rand_score
est = []
for nombre, d in D.items():
    P = d["P"]; n = len(P); rs = np.random.RandomState(SEED)
    for k in range(2, 9):
        aris = []
        for rep in range(REPETICIONES):
            a = rs.choice(n, n // 2, replace=False); b = np.setdiff1d(np.arange(n), a)
            ka = KMeans(n_clusters=k, init="k-means++", n_init=5, random_state=rep).fit(P[a])
            kb = KMeans(n_clusters=k, init="k-means++", n_init=5, random_state=1000 + rep).fit(P[b])
            aris.append(adjusted_rand_score(ka.predict(P), kb.predict(P)))
        est.append({"sujeto": nombre, "k": k, "ari_medio": round(float(np.mean(aris)), 3), "ari_peor": round(float(np.min(aris)), 3),
                    "pares_bajo_0_9": int(np.sum(np.array(aris) < 0.9)), "pares": REPETICIONES})
kmeans_estabilidad = pd.DataFrame(est)
print(kmeans_estabilidad.pivot(index="k", columns="sujeto", values=["ari_medio", "ari_peor"]).to_string())

       ari_medio           ari_peor          
sujeto   entidad proveedor  entidad proveedor
k                                            
2          0.927     1.000    0.809     1.000
3          0.777     0.990    0.441     0.976
4          0.491     0.931    0.333     0.634
5          0.815     0.984    0.533     0.958
6          0.799     0.968    0.413     0.924
7          0.718     0.634    0.566     0.541
8          0.594     0.935    0.459     0.765


In [8]:
# Perfil de cada grupo con el k elegido: mediana de cada característica (en sus unidades originales)
for nombre, d in D.items():
    cfg, k = SUJETOS[nombre], d["k"]
    p = kmeans_perfiles[(kmeans_perfiles.sujeto == nombre) & (kmeans_perfiles.k == k)]
    tabla = p.pivot(index="cluster", columns="caracteristica", values="mediana")[cfg["num"]]
    tabla.insert(0, "n", p.groupby("cluster").n.first())
    print(f"=== {nombre}, k = {k} ==="); print(tabla.round(2).to_string()); print()
    cat = d["df"].assign(cluster=d["km"][k].labels_).groupby("cluster")[cfg["desc"][0]].agg(lambda s: s.value_counts(normalize=True).head(2).round(2).to_dict())
    print(f"{cfg['desc'][0]} más frecuente por grupo:"); print(cat.to_string()); print()

=== proveedor, k = 5 ===
caracteristica      n  n_participaciones  tasa_exito  n_adjudicaciones  monto_total  monto_mediano  n_entidades  n_departamentos  concentracion  meses_activo
cluster                                                                                                                                                      
0               16177               15.0        0.00               0.0          0.0            0.0          0.0              0.0           0.00          20.1
1               11836               19.0        0.09               1.0     234662.5       117355.9          1.0              1.0           1.00          23.3
2                5492               99.0        0.08               7.0    1903037.0       146705.0          4.0              2.0           0.33          28.7
3                5670                7.0        0.00               0.0          0.0            0.0          0.0              0.0           0.00           4.0
4                3940      

## 3. Densidad: DBSCAN y HDBSCAN (sobre una muestra)
- **DBSCAN** agrupa puntos que tienen al menos `min_samples` vecinos a distancia `eps`. Los que no encajan quedan como **ruido** (etiqueta −1): son los candidatos a *outlier*.
- **HDBSCAN** es parecido pero no necesita `eps`; pide `min_cluster_size` (tamaño mínimo de un grupo).
- **Cómo se elige `eps`:** se calcula, para cada punto, la distancia a su 10.º vecino (el "gráfico de k-distancias" visto en clase) y se ordenan de menor a mayor. Donde la curva **sube de golpe** empiezan los puntos aislados. Se prueban cuatro percentiles de esa curva (50, 70, 85 y 95). `eps` pequeño → mucho ruido; grande → todo en un solo grupo. La curva se guarda para dibujarla en la web.
- La muestra es de hasta `N_MUESTRA` puntos al azar (las entidades caben todas).

In [9]:
MIN_SAMPLES = [5, 10, 20]
MIN_CLUSTER = [10, 25, 50]
grid, etiquetas = [], {}
for nombre, d in D.items():
    n = len(d["df"])
    d["idx"] = np.sort(np.random.RandomState(SEED).choice(n, min(N_MUESTRA, n), replace=False))
    Ps = d["P"][d["idx"]]
    dist10 = NearestNeighbors(n_neighbors=10).fit(Ps).kneighbors(Ps)[0][:, -1]
    d["dist10_muestra"] = dist10
    d["eps_lista"] = sorted(set(float(x) for x in np.round(np.percentile(dist10, [50, 70, 85, 95]), 2)))
    d["eps_defecto"] = float(np.round(np.percentile(dist10, 85), 2))
    et = {}
    for eps in d["eps_lista"]:
        for ms in MIN_SAMPLES:
            lab = DBSCAN(eps=eps, min_samples=ms).fit_predict(Ps)
            et[f"db_{eps:.2f}_{ms}"] = lab
            grid.append({"sujeto": nombre, "metodo": "DBSCAN", "eps": eps, "min_samples": ms,
                         "n_clusters": int(len(set(lab)) - (1 if -1 in lab else 0)), "pct_ruido": round(100 * np.mean(lab == -1), 1)})
    for mcs in MIN_CLUSTER:
        lab = HDBSCAN(min_cluster_size=mcs).fit_predict(Ps)
        et[f"hdb_{mcs}"] = lab
        grid.append({"sujeto": nombre, "metodo": "HDBSCAN", "eps": None, "min_samples": mcs,
                     "n_clusters": int(len(set(lab)) - (1 if -1 in lab else 0)), "pct_ruido": round(100 * np.mean(lab == -1), 1)})
    d["etiquetas"] = et
    print(nombre, "| muestra:", len(Ps), "| eps probados:", d["eps_lista"], "| eps por defecto (percentil 85):", d["eps_defecto"])
dbscan_grid = pd.DataFrame(grid)
dbscan_grid

/usr/local/lib/python3.13/dist-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/usr/local/lib/python3.13/dist-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/usr/local/lib/python3.13/dist-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


proveedor | muestra: 5000 | eps probados: [0.14, 0.31, 0.44, 0.67] | eps por defecto (percentil 85): 0.44


entidad | muestra: 2679 | eps probados: [0.76, 0.88, 1.07, 1.44] | eps por defecto (percentil 85): 1.07


/usr/local/lib/python3.13/dist-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/usr/local/lib/python3.13/dist-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/usr/local/lib/python3.13/dist-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,sujeto,metodo,eps,min_samples,n_clusters,pct_ruido
0,proveedor,DBSCAN,0.14,5,22,38.7
1,proveedor,DBSCAN,0.14,10,8,45.6
2,proveedor,DBSCAN,0.14,20,5,55.4
3,proveedor,DBSCAN,0.31,5,21,13.6
4,proveedor,DBSCAN,0.31,10,12,22.6
5,proveedor,DBSCAN,0.31,20,4,31.9
6,proveedor,DBSCAN,0.44,5,6,5.5
7,proveedor,DBSCAN,0.44,10,4,8.9
8,proveedor,DBSCAN,0.44,20,2,16.3
9,proveedor,DBSCAN,0.67,5,4,1.5


## 4. Jerárquico (Ward) sobre una muestra
Une los puntos de abajo hacia arriba, juntando siempre el par que menos aumenta la dispersión dentro del grupo (**Ward**). El resultado es un **dendrograma**: la altura de cada unión indica qué tan distintos eran los grupos que se juntaron. Se muestra recortado a las 25 ramas finales para que se pueda leer.

In [10]:
dendro = {}
for nombre, d in D.items():
    n = len(d["df"])
    idx_h = np.sort(np.random.RandomState(SEED + 1).choice(n, min(N_JERARQUICO, n), replace=False))
    enlace = linkage(d["P"][idx_h], method="ward")
    r = dendrogram(enlace, truncate_mode="lastp", p=25, no_plot=True)
    dendro[nombre] = {"n_puntos": int(len(idx_h)), "icoord": r["icoord"], "dcoord": r["dcoord"],
                      "hojas": [int(s.strip("()")) if s.startswith("(") else 1 for s in r["ivl"]]}
    alturas = enlace[-6:, 2].round(1)
    print(nombre, "| muestra:", len(idx_h), "| altura de las últimas 6 uniones:", alturas)

proveedor | muestra: 1500 | altura de las últimas 6 uniones: [ 20.4  24.1  36.9  40.4  46.  124.1]
entidad | muestra: 1500 | altura de las últimas 6 uniones: [29.4 33.5 45.4 47.9 56.2 77.6]


## 5. Casos extremos (outliers)
Para no depender de la muestra de 5,000 puntos, los casos extremos se buscan en **todos** los proveedores y entidades. Un punto es **caso extremo** si cumple dos cosas a la vez:
1. Está en una **zona rala**: la distancia a su 10.º vecino es de las más grandes (el 15% más alto). Es la misma idea de "ruido" de DBSCAN, pero medida con todos los puntos.
2. Está **lejos del centro de su grupo KMeans** (con el *k* elegido). Entre los que cumplen 1, se muestran los 30 más lejanos.

La columna `rasgo_extremo` dice qué característica los separa más (en desviaciones estándar respecto del promedio, después del logaritmo).

In [11]:
PCT_RALA = 85       # "zona rala" = por encima de este percentil de la distancia al 10.º vecino
filas = []
resumen_ext = {}
for nombre, d in D.items():
    cfg, df, k, P = SUJETOS[nombre], d["df"], d["k"], d["P"]
    km = d["km"][k]
    lejania = km.transform(P).min(axis=1)                                          # todos los puntos
    dist10 = NearestNeighbors(n_neighbors=10).fit(P).kneighbors(P)[0][:, -1]       # igual que min_samples = 10 en DBSCAN
    corte = float(np.percentile(dist10, PCT_RALA)); rala = dist10 > corte
    resumen_ext[nombre] = dict(n=len(P), ralos=int(rala.sum()), corte=round(corte, 2))
    cand = pd.DataFrame({"i": np.where(rala)[0], "lejania": lejania[rala]}).sort_values("lejania", ascending=False).head(30)
    for _, r in cand.iterrows():
        i = int(r.i); z = d["Z"][i]; j = int(np.argmax(np.abs(z)))
        fila = {"sujeto": nombre, "id": df.loc[i, cfg["id"]], "nombre": df.loc[i, "nombre"], "lejania": round(float(r.lejania), 2),
                "dist_vecino10": round(float(dist10[i]), 2), "rasgo_extremo": cfg["num"][j], "z": round(float(z[j]), 1),
                "cluster_kmeans": int(km.labels_[i])}
        for c in cfg["num"]:
            fila[c] = round(float(df.loc[i, c]), 3)
        for c in cfg["desc"]:
            fila[c] = df.loc[i, c]
        filas.append(fila)
outliers = proteger(pd.DataFrame(filas))
print("Puntos en zona rala (percentil %d de la distancia al 10.º vecino):" % PCT_RALA, resumen_ext)
outliers.groupby("sujeto").head(8)[["sujeto", "nombre", "rasgo_extremo", "z", "lejania"]]

Puntos en zona rala (percentil 85 de la distancia al 10.º vecino): {'proveedor': {'n': 43115, 'ralos': 6468, 'corte': 0.24}, 'entidad': {'n': 2679, 'ralos': 402, 'corte': 1.07}}


,sujeto,nombre,rasgo_extremo,z,lejania
0,proveedor,GRUPO SANTA FE SOCIEDAD ANONIMA CERRADA - GRUP...,n_adjudicaciones,7.5,8.61
1,proveedor,MAPFRE PERU COMPAÑIA DE SEGUROS Y REASEGUROS,n_entidades,6.7,7.48
2,proveedor,DIAGNOSTICA PERUANA S.A.C.,n_adjudicaciones,6.1,7.45
3,proveedor,TMI-TYC SOCIEDAD ANONIMA CERRADA - TMI-TYC S.A.C.,n_entidades,6.4,7.21
4,proveedor,AMERICA MOVIL PERU S.A.C.,n_entidades,6.5,7.14
5,proveedor,LA POSITIVA SEGUROS Y REASEGUROS S.A.,n_entidades,6.4,6.94
6,proveedor,LINDE PERU S.R.L.,n_entidades,5.1,6.82
7,proveedor,LABORATORIOS AC FARMA S.A.,n_adjudicaciones,5.5,6.65
30,entidad,ORGANISMO DE EVALUACION Y FISCALIZACION AMBIENTAL,n_procesos,4.1,6.00
31,entidad,PROYECTO DE MODERNIZACION DE LA GESTION DE LOS...,concentracion,3.6,5.50


## 6. Archivos para la página web

In [12]:
filas = []
for nombre, d in D.items():
    cfg, df, idx = SUJETOS[nombre], d["df"], d["idx"]
    p = pd.DataFrame({"sujeto": nombre, "id": df.loc[idx, cfg["id"]].values, "nombre": df.loc[idx, "nombre"].values,
                      "cp1": d["P"][idx, 0].round(3), "cp2": d["P"][idx, 1].round(3)})
    for k in KS:
        p[f"km_{k}"] = d["km"][k].labels_[idx]
    for col, lab in d["etiquetas"].items():
        p[col] = lab
    filas.append(p)
puntos = pd.concat(filas, ignore_index=True)

guardar_csv("pca_varianza.csv", pca_varianza)
guardar_csv("kmeans_k.csv", kmeans_k)
guardar_csv("kmeans_perfiles.csv", kmeans_perfiles)
guardar_csv("puntos.csv", puntos)
guardar_csv("dbscan_grid.csv", dbscan_grid)
guardar_csv("kmeans_estabilidad.csv", kmeans_estabilidad)
kd = []
for nombre, d in D.items():
    for p in range(0, 101):
        kd.append({"sujeto": nombre, "percentil": p, "dist_vecino10": round(float(np.percentile(d["dist10_muestra"], p)), 3)})
guardar_csv("dbscan_kdist.csv", pd.DataFrame(kd))
guardar_csv("outliers.csv", outliers)
with open(f"{WEB}/dendrograma.json", "w", encoding="utf-8") as f:
    json.dump(dendro, f)
print("guardado: dendrograma.json")

guardado: pca_varianza.csv 9 filas
guardado: kmeans_k.csv 18 filas
guardado: kmeans_perfiles.csv 972 filas
guardado: puntos.csv 7679 filas
guardado: dbscan_grid.csv 30 filas
guardado: kmeans_estabilidad.csv 14 filas


guardado: dbscan_kdist.csv 202 filas
guardado: outliers.csv 60 filas
guardado: dendrograma.json


## 7. Lectura y respuesta a P1

**PCA**
- **Proveedores:** 4 componentes explican 95%. La primera (64%) es el *tamaño de lo adjudicado* (monto, adjudicaciones, entidades y departamentos pesan parecido). La segunda (16%) separa a los que **participan mucho** de los que **ganan mucho** (participaciones y meses activos suben, la tasa de éxito baja).
- **Entidades:** 5 componentes explican 96%. La primera (42%) es el *tamaño* (proveedores, procesos, monto). La segunda (18%) opone **bienes** y **servicios**.

**KMeans++ — proveedores (*k* = 5).** Con *k* = 2 la silueta es la más alta (0.597), pero solo separa a los que **ganan** de los que **no ganan ninguna vez**. Con *k* = 5 salen cinco grupos, que forman **tres tipos**:

| Grupo | Proveedores | Perfil (medianas) |
|---|---|---|
| 3 · **Pasaron de largo** | 5,670 (13%) | 7 participaciones, 0 adjudicaciones, solo 4 meses activos |
| 0 · **Insisten y no ganan** | 16,177 (38%) | 15 participaciones, 0 adjudicaciones, 20 meses activos |
| 1 · **Ganan poco con una sola entidad** | 11,836 (27%) | 19 participaciones, 1 adjudicación (S/ 234,663), tasa de éxito 0.09, concentración 1.0 |
| 4 · **Pocas participaciones y ganan bastante de ellas** | 3,940 (9%) | 8 participaciones, tasa de éxito 0.42, 4 adjudicaciones (S/ 690,166), 2 entidades |
| 2 · **Activos en varias entidades** | 5,492 (13%) | 99 participaciones, 7 adjudicaciones (S/ 1.90 millones), 4 entidades y 2 departamentos, concentración 0.33 |

Los tres tipos son: **no ganan** (grupos 3 y 0: 21,847 proveedores, 50.7%), **ganan poco** (grupos 1 y 4: 15,776, 36.6%) y **activos en varias entidades** (grupo 2: 5,492, 12.7%). Los grupos mezclan bienes y servicios: los que no ganan se inclinan a servicios (50 a 54% frente a 27 a 31% de bienes) y los activos y el grupo de éxito alto, a bienes (53 y 58%). Lo que más los diferencia es **cuánto ganan, con cuántas entidades y por cuánto tiempo participan**, no el rubro.

**¿Por qué *k* = 5?** El codo de la inercia no es nítido, así que se miraron tres cosas:
- **Codo:** la inercia baja 37,264 al pasar de 2 a 3 grupos, 21,656 de 3 a 4, 19,822 de 4 a 5 y solo 9,656 de 5 a 6. La mejora se frena **después de *k* = 3** (el paso a 4 mejora 58% de lo anterior), casi no se frena al pasar a 5 (92%) y **se frena fuerte después de *k* = 5** (el paso a 6 mejora 49%). Entre 3 y 5 hay una meseta y *k* = 5 es el último antes de la caída fuerte.
- **Silueta:** la mayor desde *k* = 3 es 0.536 (en *k* = 3). Baja a 0.463 con *k* = 4 y queda igual con *k* = 5 (0.462): pasar de 4 a 5 no empeora la separación.
- **Estabilidad (ARI, sección 2c):** con *k* = 5 los grupos salen casi iguales aunque cambien los datos (ARI promedio 0.984; ninguna de 20 pruebas bajo 0.9). Con *k* = 4 es el más bajo de 3 a 6 (0.931; 3 de 20 pruebas dan otra partición, la peor con 0.634): partir «no ganan» por meses activos es un corte dentro de un continuo y a veces el algoritmo lo hace en otro sitio. Con *k* = 3 es 0.990.
- **Sensibilidad:** con *k* = 3, 4 o 5 los extremos son los mismos («no ganan» y «activos en varias entidades»); lo que cambia es el detalle. Con *k* = 4 «no ganan» se parte en 5,670 y 16,177. Con *k* = 5 además «ganan poco» se parte en 11,836 y un grupo pequeño de 3,940 con tasa de éxito 0.42. Como no hay un único *k* correcto (los datos forman un continuo: HDBSCAN también halla un solo núcleo denso grande), se eligió el que cumple los tres criterios a la vez.

**KMeans++ — entidades** (*k* = 6, silueta 0.25 y estabilidad 0.80: **ambas bajas**; 16 de 20 pruebas dieron grupos distintos con otra mitad de datos). Los grupos se solapan y son **tendencias, no fronteras**. Con *k* = 5 el resultado es parecido (silueta 0.251, estabilidad 0.82), así que la elección entre 5 y 6 casi no importa; se dejó 6 por tener la mejor silueta.
- 1 · **Grandes y diversas** (717): mediana de 132 procesos y 56 proveedores.
- 0 · **Municipalidades con obras** (647): 43% de sus procesos son obras.
- 5 · **Compran sobre todo bienes** (560): 66% bienes.
- 2 · **Compran sobre todo servicios** (426): 81% servicios.
- 4 · **Pocas compras, muy concentradas** (297): 9 procesos y concentración 0.50.
- 3 · **Sin adjudicaciones registradas** (32): 7 procesos y ningún proveedor adjudicado.

**Densidad.** Con `eps` en el percentil 85 y `min_samples = 10`, DBSCAN encuentra 4 grupos y 8.9% de ruido en proveedores (3 grupos y 6.0% en entidades). HDBSCAN con grupos mínimos de 10 encuentra 5 grupos y 5.4% de ruido en proveedores, pero con 25 o más solo ve **2 grupos**: los datos tienen un núcleo denso grande y los tipos finos del KMeans son más bien zonas dentro de él. La curva de distancias al 10.º vecino (gráfico de k-distancias, guardada en `dbscan_kdist.csv`) sube de golpe en su último tramo: ahí empiezan los puntos aislados. Al bajar `eps` el ruido sube muy rápido (de 8.9% a 22.6% en proveedores al pasar del percentil 85 al 70).

**Dendrograma.** En proveedores la última unión sube a 124, mucho más que las anteriores (20 a 46): hay **dos grandes bloques** (los que ganan y los que no), igual que con *k* = 2. En entidades las alturas crecen de forma gradual (29 a 78), lo que concuerda con la silueta baja.

**Casos extremos** (zona rala y lejos del centro de su grupo, buscados en **todos** los puntos y no solo en la muestra: 6,468 proveedores y 402 entidades caen en zona rala).
- De 30 proveedores, **29 son del grupo «activos en varias entidades»**: son **muy grandes y generalistas**. Por ejemplo GRUPO SANTA FE (1,427 adjudicaciones con 262 entidades), MAPFRE PERÚ (314 adjudicaciones con 173 entidades) o TMI-TYC (255 adjudicaciones con 140 entidades). 20 salen por el número de entidades, 5 por adjudicaciones y 3 por departamentos.
- Solo 2 salen por una tasa de éxito altísima, y con muchas participaciones: SISTEMAS ORACLE DEL PERÚ (ganó 90 de 90 procesos, 48 entidades) y TELEMÁTICA (74 participaciones, tasa 0.92). Con tantas participaciones ya no se explica por azar; puede ser un nicho con pocos competidores y requiere mirar el rubro.
- En entidades salen las **muy grandes** (OEFA con 3,582 procesos, Petroperú con 2,228, PROMPERÚ con 1,720), los **proyectos con un solo proveedor** (concentración 1.0 con 6 procesos) y las que tienen **muy pocos postores** por proceso (menos de 1.5 en promedio) o **demasiados** (Instituto Vial Provincial de Andahuaylas, 84). El rasgo extremo es el promedio de postores en 19 de 30.

**Respuesta a P1.** Los proveedores se reparten en **tres tipos** según su relación con el Estado: quienes **no ganan**, quienes **ganan poco** y quienes son **activos en varias entidades**. Con *k* = 5 se afinan dos: entre quienes no ganan se separan los que **pasaron de largo** (4 meses) de los que **insisten** (20 meses), y entre quienes ganan poco aparece un grupo pequeño con tasa de éxito alta y pocas participaciones. Se salen del tipo los proveedores enormes y generalistas.

**Límites.**
- Los consorcios se dejaron fuera, así que un proveedor que gana **solo dentro de consorcios** aparece como "no gana".
- Los datos cubren 2.7 años; un proveedor "nuevo" puede verse como de una sola entidad solo por falta de historia.
- Un caso extremo es **una pista para revisar**, no un indicio de irregularidad: un proveedor grande y un proveedor de nicho se ven igual de raros para el algoritmo.